### CARGA DE LIBRERÍAS NECESARIAS PARA LA LIMPIEZA


In [1]:
import pandas as pd
import numpy as np
import os

### CARGA DEL ARCHIVO A LIMPIAR

In [2]:
Ruta_Archivo_Entrada = '../Original_Data/2025_3T_Flujosportipodeinversion_actu__3_.xlsx'
Ruta_Archivo_Salida  = '../Prepared_Data/General'
Archivo_Excel        = pd.read_excel(Ruta_Archivo_Entrada, sheet_name = None, header = None)

In [3]:
'''Ubicación del entorno de trabajo y hojas de excel contenidas dentro del archivo'''
#print(os.getcwd())
Archivo_Excel.keys()

dict_keys(['Por tipo de inversión', 'Por origen', 'Por sector', 'Por entidad federativa'])

### LIMPIEZA DE LA PRIMERA HOJA DEL ARCHIVO: 'Por tipo de inversión'

##### Selección del archivo a limpiar 

In [4]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Invesment_Types = Archivo_Excel['Por tipo de inversión']
#display(Invesment_Types)
Invesment_Types_Description = Invesment_Types.iloc[0,0]
#print(Invesment_Types_Description)
Invesment_Types_Database = Invesment_Types.iloc[-1,0]
#print(Invesment_Types_Database)

##### Limpieza general de dataframe

In [5]:
'''Se crea una copia del archivo con el cual trabajaremos '''
Invesment_Types_1 = Invesment_Types.copy()
''' Se elimninan las dos primeras filas y las tres últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Invesment_Types_1 = Invesment_Types_1.iloc[2:,:].reset_index(drop=True)
Invesment_Types_1 = Invesment_Types_1.iloc[:-3,:].reset_index(drop=True)
'''Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Invesment_Types_1.iloc[0,1:] = Invesment_Types_1.iloc[0,1:].ffill() 
''' Se extraen los años contenidos en la fila 0 '''
Invesment_Types_1_Años = Invesment_Types_1.iloc[0,1:].astype(int).tolist()
#print(Invesment_Types_1_Años)
''' Se extraen los trimestres contenidos en la fila 1 '''
Invesment_Types_1_Trimestres = Invesment_Types_1.iloc[1,1:].astype(int).tolist()
#print(Invesment_Types_1_Trimestres)
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo en formato adecuado '''
Invesment_Types_1_Fecha = [f"{a}Q{t}" for a,t in zip(Invesment_Types_1_Años,Invesment_Types_1_Trimestres)]
#print(Invesment_Types_1_Fecha)
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Invesment_Types_1.columns = [Invesment_Types_1.iloc[0,0]] + Invesment_Types_1_Fecha
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados'''
Invesment_Types_1 = Invesment_Types_1.iloc[2:,:].reset_index(drop=True)
#display(Invesment_Types_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_12912\681498941.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Invesment_Types_1.iloc[0,1:] = Invesment_Types_1.iloc[0,1:].ffill()


##### Transformación de dataframe

In [6]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal además de un análisis categórico relativamente sencillo'''
Invesment_Types_2 = pd.melt(
    Invesment_Types_1,
    id_vars = ['Tipo de Inversión'],
    var_name = 'Año_Trimestre',
    value_name = 'IED'
)
'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Invesment_Types_2['Fecha'] = pd.PeriodIndex(Invesment_Types_2['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Invesment_Types_2 = Invesment_Types_2.drop(['Año_Trimestre'],axis=1)
display(Invesment_Types_2)

,Tipo de Inversión,IED,Fecha
0,Total general,7437.856436,2006-03-31
1,Nuevas inversiones,897.606260,2006-03-31
2,Reinversión de utilidades,4839.453874,2006-03-31
3,Cuentas entre compañías,1700.796301,2006-03-31
4,Total general,14072.167427,2006-06-30
...,...,...,...
307,Cuentas entre compañías,2317.956853,2025-03-31
308,Total general,33729.676890,2025-06-30
309,Nuevas inversiones,3266.823851,2025-06-30
310,Reinversión de utilidades,28798.405921,2025-06-30


##### Formato general del archivo

In [7]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Invesment_Types_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Tipo de Inversión  312 non-null    object        
 1   IED                312 non-null    float64       
 2   Fecha              312 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 7.4+ KB


##### Carga de archivo al entorno local

In [8]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_1 = 'Investment_Types.csv'
Ruta_Salida_1 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_1)
Invesment_Types_2.to_csv(Ruta_Salida_1, index=False, encoding='utf-8-sig')

### LIMPIEZA DE LA SEGUNDAHOJA DEL ARCHIVO: 'Por origen'

##### Selección del archivo a limpiar 

In [9]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Origin_Investment_by_Country = Archivo_Excel['Por origen']
#display(Origin_Investment_by_Country)
Origin_Investment_by_Country_Description = Origin_Investment_by_Country.iloc[0,0]
#print(Origin_Investment_by_Country_Description)
Origin_Investment_by_Country_Database = Origin_Investment_by_Country.iloc[-1,0]
#print(Origin_Investment_by_Country_Database)

##### Limpieza general de dataframe

In [10]:
'''Se crea una copia del archivo con el cual trabajaremos '''
Origin_Investment_by_Country_1 = Origin_Investment_by_Country.copy()
''' Se elimninan las dos primeras filas y las tres últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Origin_Investment_by_Country_1 = Origin_Investment_by_Country_1.iloc[2:,:].reset_index(drop=True)
Origin_Investment_by_Country_1 = Origin_Investment_by_Country_1.iloc[:-3,:].reset_index(drop=True)
'''Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Origin_Investment_by_Country_1.iloc[0,1:] = Origin_Investment_by_Country_1.iloc[0,1:].ffill()
''' Se extraen los años contenidos en la fila 0 '''
Origin_Investment_by_Country_1_Años = Origin_Investment_by_Country_1.iloc[0,1:].astype(int).tolist()
#print(Origin_Investment_by_Country_1_Años)
''' Se extraen los trimestres contenidos en la fila 1 '''
Origin_Investment_by_Country_1_Trimestres = Origin_Investment_by_Country_1.iloc[1,1:].astype(int).tolist()
#print(Origin_Investment_by_Country_1_Trimestres)
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo en formato adecuado '''
Origin_Investment_by_Country_1_Fecha = [f"{a}Q{t}" for a,t in zip(Origin_Investment_by_Country_1_Años,Origin_Investment_by_Country_1_Trimestres)]
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Origin_Investment_by_Country_1.columns = [Origin_Investment_by_Country_1.iloc[0,0]] + Origin_Investment_by_Country_1_Fecha
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados'''
Origin_Investment_by_Country_1 = Origin_Investment_by_Country_1.iloc[2:,:].reset_index(drop=True)
#display(Origin_Investment_by_Country_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_12912\1486416111.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Origin_Investment_by_Country_1.iloc[0,1:] = Origin_Investment_by_Country_1.iloc[0,1:].ffill()


##### Transformación de dataframe

In [11]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal además de un análisis categórico relativamente sencillo'''
Origin_Investment_by_Country_2 = pd.melt(
    Origin_Investment_by_Country_1,
    id_vars = ['Origen'],
    var_name = 'Año_Trimestre',
    value_name = 'IED'
)
'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Origin_Investment_by_Country_2['Fecha'] = pd.PeriodIndex(Origin_Investment_by_Country_2['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Origin_Investment_by_Country_2 = Origin_Investment_by_Country_2.drop(['Año_Trimestre'],axis=1)
display(Origin_Investment_by_Country_2)

,Origen,IED,Fecha
0,Total general,7437.856436,2006-03-31
1,Afganistán,0,2006-03-31
2,Albania,0,2006-03-31
3,Alemania,165.425837,2006-03-31
4,Andorra,0,2006-03-31
...,...,...,...
11851,Vietnam,C,2025-06-30
11852,"Yemen, República de",0,2025-06-30
11853,Zambia,0,2025-06-30
11854,Zimbabue,0,2025-06-30


##### Formato general del archivo

In [12]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Origin_Investment_by_Country_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11856 entries, 0 to 11855
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Origen  11856 non-null  object        
 1   IED     11856 non-null  object        
 2   Fecha   11856 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(2)
memory usage: 278.0+ KB


##### Carga de archivo al entorno local

In [13]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_2 = 'Origin_of_Investment.csv'
Ruta_Salida_2 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_2)
Origin_Investment_by_Country_2.to_csv(Ruta_Salida_2, index=False, encoding='utf-8-sig')

### LIMPIEZA DE LA TERCERA HOJA DEL ARCHIVO: 'Por sector'


##### Selección del archivo a limpiar 

In [14]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Investment_by_Sector = Archivo_Excel['Por sector']
#display(Investment_by_Sector)
Investment_by_Sector_Description = Investment_by_Sector.iloc[0,0]
#print(Investment_by_Sector_Description)
Investment_by_Sector_Database = Investment_by_Sector.iloc[-1,0]
#print(Investment_by_Sector_Database)

##### Limpieza general de dataframe

In [15]:
''' Se crea una copia del archivo con el cual trabajaremos '''
Investment_by_Sector_1 = Investment_by_Sector.copy()
''' Se elimninan las dos primeras filas y las cinco últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Investment_by_Sector_1 = Investment_by_Sector_1.iloc[2:,:].reset_index(drop=True)
Investment_by_Sector_1 = Investment_by_Sector_1.iloc[:-5,:].reset_index(drop=True)
''' Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Investment_by_Sector_1.iloc[0,1:] = Investment_by_Sector_1.iloc[0,1:].ffill()
''' Se extraen los años contenidos en la fila 0 '''
Investment_by_Sector_1_Años = Investment_by_Sector_1.iloc[0,1:].astype(int).tolist()
#print(Investment_by_Sector_1_Años)
''' Se extraen los trimestres contenidos en la fila 1 '''
Investment_by_Sector_1_Trimestres = Investment_by_Sector_1.iloc[1,1:].astype(int).tolist()
#print(Investment_by_Sector_1_Trimestres)
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo en formato adecuado '''
Investment_by_Sector_1_Fecha = [f"{a}Q{t}" for a, t in zip(Investment_by_Sector_1_Años, Investment_by_Sector_1_Trimestres)]
#print(Investment_by_Sector_1_Fecha)
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Investment_by_Sector_1.columns = [Investment_by_Sector_1.iloc[0,0]] + Investment_by_Sector_1_Fecha
#display(Investment_by_Sector_1)
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados'''
Investment_by_Sector_1 = Investment_by_Sector_1.iloc[2:,:].reset_index(drop=True)
#display(Investment_by_Sector_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_12912\3435403883.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Investment_by_Sector_1.iloc[0,1:] = Investment_by_Sector_1.iloc[0,1:].ffill()


##### Ingeniería de datos, métodos de categorización y jerarquización 

In [16]:
'''A continuación se crea una copia del archivo con el cual trabajaremos '''
Investment_by_Sector_2 = Investment_by_Sector_1.copy()
'''Definimos tanto el nivel como el sector, subsector y rama al que pertenecen los diferentes conceptos dentro de nuestro dataset '''
Investment_by_Sector_2[['Código','Concepto']] = Investment_by_Sector_2['Sector, subsector y rama'].str.split(r"\s+", n=1, expand=True)
Investment_by_Sector_2['Nivel'] = Investment_by_Sector_2['Código'].apply(lambda x: 2 if '-' in x else len(x))
Investment_by_Sector_2['Sector'] = Investment_by_Sector_2['Código'].apply(lambda x: x if '-' in x else x[:2] )
Investment_by_Sector_2['Subsector'] = Investment_by_Sector_2['Código'].apply(lambda x: None if '-' in x or len(x)<3 else x[:3])
Investment_by_Sector_2['Rama'] = Investment_by_Sector_2['Código'].apply(lambda x: None if '-' in x or len(x)<4 else x[:4])
'''Dados los ajustes anteriores la fila en específico para el Total general ha quedado en un formato no deseado, por lo que, se corrige '''
Investment_by_Sector_2.loc[0,'Concepto'] = 'Total general'
Investment_by_Sector_2.loc[0,'Nivel':] = 1
'''Se eliminan las columnas inncesarias '''
Investment_by_Sector_2 = Investment_by_Sector_2.drop(['Sector, subsector y rama','Código'],axis=1)
#display(Investment_by_Sector_2)

##### Transformación del dataframe

In [17]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal además de un análisis categórico relativamente sencillo'''
Investment_by_Sector_3 = pd.melt(
    Investment_by_Sector_2,
    id_vars = ['Concepto','Nivel','Sector','Subsector','Rama'],
    var_name = 'Año_Trimestre',
    value_name = 'IED'
)
'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Investment_by_Sector_3['Fecha'] = pd.PeriodIndex(Investment_by_Sector_3['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Investment_by_Sector_3 = Investment_by_Sector_3.drop(['Año_Trimestre'],axis=1)
Investment_by_Sector_3[['Nivel']] = Investment_by_Sector_3[['Nivel']].astype(int)
display(Investment_by_Sector_3)

,Concepto,Nivel,Sector,Subsector,Rama,IED,Fecha
0,Total general,1,1,1,1,7437.856436,2006-03-31
1,"Agricultura, cría y explotación de animales, a...",2,11,None,None,-3.478962,2006-03-31
2,Agricultura,3,11,111,None,-3.591288,2006-03-31
3,"Cultivo de semillas oleaginosas, leguminosas y...",4,11,111,1111,-0.818922,2006-03-31
4,Cultivo de hortalizas,4,11,111,1112,-2.821496,2006-03-31
...,...,...,...,...,...,...,...
30805,Asociaciones y organizaciones,3,81,813,None,C,2025-06-30
30806,"Asociaciones y organizaciones comerciales, lab...",4,81,813,8131,C,2025-06-30
30807,"Asociaciones y organizaciones religiosas, polí...",4,81,813,8132,0,2025-06-30
30808,Hogares con empleados domésticos,3,81,814,None,0,2025-06-30


##### Formato general del archivo

In [18]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Investment_by_Sector_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30810 entries, 0 to 30809
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Concepto   30810 non-null  object        
 1   Nivel      30810 non-null  int64         
 2   Sector     30810 non-null  object        
 3   Subsector  29328 non-null  object        
 4   Rama       22308 non-null  object        
 5   IED        30810 non-null  object        
 6   Fecha      30810 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(5)
memory usage: 1.6+ MB


##### Carga de archivo al entorno local

In [19]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_3 = 'Investment_by_Sector.csv'
Ruta_Salida_3 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_3)
Investment_by_Sector_3.to_csv(Ruta_Salida_3, index=False, encoding='utf-8-sig')

### LIMPIEZA DE LA CUARTA HOJA DEL ARCHIVO: 'Por entidad federativa'


##### Selección del archivo a limpiar 

In [20]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Origin_Investment_by_State = Archivo_Excel['Por entidad federativa']
display(Origin_Investment_by_State)
Origin_Investment_by_State_Description = Origin_Investment_by_State.iloc[0,0]
#print(Origin_Investment_by_State_Description)
Origin_Investment_by_State_Database = Origin_Investment_by_State.iloc[-1,0]
#print(Origin_Investment_by_State_Database)

,0,1,2,3,4,5,6,7,8,9,...,69,70,71,72,73,74,75,76,77,78
0,Inversión Extranjera Directa en México por ent...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Entidad federativa,2006.000000,NaN,NaN,NaN,2007.000000,NaN,NaN,NaN,2008.000000,...,2023.000000,NaN,NaN,NaN,2024.000000,NaN,NaN,NaN,2025.000000,NaN
3,NaN,1.000000,2.000000,3.000000,4.000000,1.000000,2.000000,3.000000,4.000000,1.000000,...,1.000000,2.000000,3.000000,4.000000,1.000000,2.000000,3.000000,4.000000,1.000000,2.000000
4,Total general,7437.856436,14072.167427,16418.890363,21236.848947,10815.781411,16953.419715,24582.073303,32394.037818,8554.831515,...,23669.096448,32204.212803,35233.461344,36513.508312,27164.486911,33633.546566,36704.201571,37893.547479,23277.457802,33729.676890
5,Aguascalientes,49.937659,90.211506,114.055341,140.593914,51.873918,142.457447,210.601066,410.573807,312.552321,...,170.672403,1199.713211,1630.280956,1080.256635,190.025948,559.186612,888.353270,1018.698497,155.069387,418.867421
6,Baja California,305.072982,619.493575,947.561103,1340.717934,488.419705,749.439163,1616.752504,1790.829226,324.301723,...,953.466944,1526.334657,1182.102636,1444.632588,1319.540044,2115.739739,2252.828795,2468.434227,921.004501,1549.283032
7,Baja California Sur,266.401903,411.866632,534.571122,592.178636,213.263629,408.186878,543.847963,819.291072,224.484518,...,206.780724,677.139481,930.297608,1146.905885,229.189129,624.252319,1200.305080,1418.166831,218.136877,604.648745
8,Campeche,15.215796,25.143849,34.305706,30.419141,12.388787,73.090144,151.326491,134.523132,13.404481,...,17.150677,-14.751556,46.320419,98.612662,47.140966,126.887116,106.186778,517.821238,-2.604194,30.898086
9,Chiapas,34.478525,62.609874,71.485173,89.256227,79.718061,102.001322,114.862621,186.876594,47.766641,...,60.912363,52.747885,44.130290,31.984403,81.856909,101.284197,105.818942,107.123121,1.686771,23.565465


##### Limpieza general de dataframe

In [21]:
''' Se crea una copia del archivo con el cual trabajaremos '''
Origin_Investment_by_State_1 = Origin_Investment_by_State.copy()
''' Se elimninan las dos primeras filas y las tres últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Origin_Investment_by_State_1 = Origin_Investment_by_State_1.iloc[2:,:].reset_index(drop=True)
Origin_Investment_by_State_1 = Origin_Investment_by_State_1.iloc[:-3,:].reset_index(drop=True)
''' Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Origin_Investment_by_State_1.iloc[0,1:] = Origin_Investment_by_State_1.iloc[0,1:].ffill()
''' Se extraen los años contenidos en la fila 0 '''
Origin_Investment_by_State_1_Año = Origin_Investment_by_State_1.iloc[0,1:].astype(int).tolist()
#print(Origin_Investment_by_State_1_Año)
''' Se extraen los trimestres contenidos en la fila 1 '''
Origin_Investment_by_State_1_Trimestres = Origin_Investment_by_State_1.iloc[1,1:].astype(int).tolist()
#print(Origin_Investment_by_State_1_Trimestres)
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo en formato adecuado '''
Origin_Investment_by_State_1_Fecha = [f"{a}Q{t}" for a,t in zip(Origin_Investment_by_State_1_Año,Origin_Investment_by_State_1_Trimestres)]
#print(Origin_Investment_by_State_1_Fecha)
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Origin_Investment_by_State_1.columns = [Origin_Investment_by_State_1.iloc[0,0]] + Origin_Investment_by_State_1_Fecha
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados'''
Origin_Investment_by_State_1 = Origin_Investment_by_State_1.iloc[2:,:].reset_index(drop=True)
#display(Origin_Investment_by_State_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_12912\479119695.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Origin_Investment_by_State_1.iloc[0,1:] = Origin_Investment_by_State_1.iloc[0,1:].ffill()


##### Transformación del dataframe

In [22]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal además de un análisis categórico relativamente sencillo'''
Origin_Investment_by_State_2 = pd.melt(
    Origin_Investment_by_State_1,
    id_vars = ['Entidad federativa'],
    var_name = 'Año_Trimestre',
    value_name = 'IED'
)
'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Origin_Investment_by_State_2['Fecha'] = pd.PeriodIndex(Origin_Investment_by_State_2['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Origin_Investment_by_State_2 = Origin_Investment_by_State_2.drop(['Año_Trimestre'],axis=1)
display(Origin_Investment_by_State_2)

,Entidad federativa,IED,Fecha
0,Total general,7437.856436,2006-03-31
1,Aguascalientes,49.937659,2006-03-31
2,Baja California,305.072982,2006-03-31
3,Baja California Sur,266.401903,2006-03-31
4,Campeche,15.215796,2006-03-31
...,...,...,...
2569,Tamaulipas,430.198706,2025-06-30
2570,Tlaxcala,96.837422,2025-06-30
2571,Veracruz de Ignacio de la Llave,63.715517,2025-06-30
2572,Yucatán,32.179340,2025-06-30


##### Formato general del archivo

In [23]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Origin_Investment_by_State_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2574 entries, 0 to 2573
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Entidad federativa  2574 non-null   object        
 1   IED                 2574 non-null   float64       
 2   Fecha               2574 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 60.5+ KB


##### Carga de archivo al entorno local

In [24]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_4 = 'Investment_by_State.csv'
Ruta_Salida_3 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_4)
Origin_Investment_by_State_2.to_csv(Ruta_Salida_3, index=False, encoding='utf-8-sig')